In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
from sodapy import Socrata
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

Directorio de trabajo : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data
Carpeta web/data      : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data


In [2]:
pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install sodapy

Note: you may need to restart the kernel to use updated packages.


## Datos diarios – Extracción API y cruce con alertas históricas

In [4]:
# Carga alertas históricas generadas por datos precipitacion historicos.ipynb
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta = pd.read_csv(out_estaciones)

print(f"Estaciones históricas cargadas: {len(estaciones_alerta):,}")
print("\n=== Distribución de alerta compuesta (histórica) ===")
estaciones_alerta.head()

Estaciones históricas cargadas: 3,053

=== Distribución de alerta compuesta (histórica) ===


,CodigoEstacion,NombreEstacion,Departamento,Municipio,Latitud,Longitud,frecuencia_extremos,pendiente,p_valor,tendencia,frecuencia_reciente,ratio_reciente,alerta_lluvias,sequia_categoria,cod_norm
0,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,5.412000,-76.418000,0.060748,NaN,NaN,NaN,NaN,NaN,BAJA,NORMAL,11017020
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,5.888719,-76.145167,0.055305,NaN,NaN,NaN,NaN,NaN,BAJA,NORMAL,11025501
2,11027030,EL SIETE,CHOCO,EL CARMEN,5.862000,-76.152056,0.033708,-0.000084,0.983976,estable,0.042003,1.246102,MODERADA,MODERADA,11027030
3,11027030,EL SIETE - AUT,CHOCO,EL CARMEN,5.862000,-76.152056,0.102041,-0.000084,0.983976,estable,0.042003,0.411634,ALTA,MODERADA,11027030
4,11027070,BORAUDO,CHOCÓ,LLORÓ,5.515000,-76.576000,0.051806,NaN,NaN,NaN,NaN,NaN,BAJA,NORMAL,11027070


In [5]:
# Carga davipola y proyecta a EPSG 3116 (necesario para sjoin con municipios)
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

print(f"Municipios cargados: {len(gdf_mun):,}")

Municipios cargados: 1,121


## Datos en tiempo real – API IDEAM

In [6]:
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)

# ORDER BY DESC es más confiable que max() para timestamps en Socrata
latest = client.get(DATASET_ID, select="fechaobservacion", order="fechaobservacion DESC", limit=1)
fecha_mapa = latest[0]["fechaobservacion"][:10]
print(f"Última fecha disponible: {fecha_mapa}")

where = (
    f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
    f"AND fechaobservacion < '{fecha_mapa}T23:59:59.999'"
)

records, offset = [], 0
while True:
    batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
    if not batch:
        break
    records.extend(batch)
    offset += 100_000
    print(f"  {len(records):,} registros descargados...")
client.close()

Última fecha disponible: 2026-04-05


  100,000 registros descargados...


  144,015 registros descargados...


In [7]:
estaciones_alerta.columns

Index(['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio',
       'Latitud', 'Longitud', 'frecuencia_extremos', 'pendiente', 'p_valor',
       'tendencia', 'frecuencia_reciente', 'ratio_reciente', 'alerta_lluvias',
       'sequia_categoria', 'cod_norm'],
      dtype='object')

In [ ]:
# Agrega lecturas a nivel de estación
df_api = pd.DataFrame.from_records(records)

for col in ("valorobservado", "latitud", "longitud"):
    df_api[col] = pd.to_numeric(df_api[col], errors="coerce")

df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
df_api = df_api[df_api["valorobservado"] >= 0]

STATION_COLS_API = [
    "codigoestacion", "nombreestacion", "departamento",
    "municipio", "zonahidrografica", "latitud", "longitud",
]

df_dia_hoy = (
    df_api.groupby(STATION_COLS_API, as_index=False)
    .agg(
        precip_acum_mm=("valorobservado", "sum"),
        precip_max_10min=("valorobservado", "max"),
        n_lecturas=("valorobservado", "count"),
    )
)
del df_api

# Normaliza cod_norm en ambos lados
df_dia_hoy["cod_norm"] = df_dia_hoy["codigoestacion"].astype(str).str.strip().str.lstrip("0")
estaciones_alerta["cod_norm"] = estaciones_alerta["cod_norm"].astype(str).str.strip().str.lstrip("0")

# ── Base = TODAS las estaciones históricas; left join con datos del día ───────
cols_precip = ['cod_norm', 'precip_acum_mm', 'precip_max_10min', 'n_lecturas', 'zonahidrografica']

df_dia = (
    estaciones_alerta
    .rename(columns={
        'CodigoEstacion': 'codigoestacion',
        'NombreEstacion': 'nombreestacion',
        'Latitud':        'latitud',
        'Longitud':       'longitud',
        'Departamento':   'departamento',
        'Municipio':      'municipio',
    })
    .merge(df_dia_hoy[cols_precip], on='cod_norm', how='left')
)
del df_dia_hoy

# Rellenar faltantes (estaciones sin lectura hoy conservan sus indicadores históricos)
df_dia["alerta_lluvias"]      = df_dia["alerta_lluvias"].fillna("BAJA")
df_dia["frecuencia_extremos"] = df_dia["frecuencia_extremos"].fillna(0)
df_dia["frecuencia_reciente"] = df_dia["frecuencia_reciente"].fillna(0)
df_dia["ratio_reciente"]      = df_dia["ratio_reciente"].fillna(np.nan)
df_dia["tendencia"]           = df_dia["tendencia"].fillna("sin_datos")
df_dia["sequia_categoria"]    = df_dia["sequia_categoria"].fillna("NORMAL")
df_dia["precip_acum_mm"]      = df_dia["precip_acum_mm"].fillna(np.nan)
df_dia["precip_max_10min"]    = df_dia["precip_max_10min"].fillna(np.nan)
df_dia["n_lecturas"]          = df_dia["n_lecturas"].fillna(0).astype(int)

print(f"Estaciones históricas (base)   : {len(df_dia):,}")
print(f"  Con lectura hoy               : {(df_dia['n_lecturas'] > 0).sum():,}")
print(f"  Sin lectura hoy               : {(df_dia['n_lecturas'] == 0).sum():,}")

In [9]:
df_dia.head()

,codigoestacion,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,precip_acum_mm,precip_max_10min,n_lecturas,cod_norm,alerta_lluvias,frecuencia_extremos,frecuencia_reciente,ratio_reciente,tendencia,sequia_categoria
0,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,3.4,0.4,144,11027030,MODERADA,0.033708,0.042003,1.246102,estable,MODERADA
1,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,3.4,0.4,144,11027030,ALTA,0.102041,0.042003,0.411634,estable,MODERADA
2,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,3.4,0.4,144,11027030,BAJA,0.000000,0.000000,NaN,insuficiente,NORMAL
3,0011027030,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862,-76.152056,3.4,0.4,144,11027030,BAJA,0.000000,0.000000,NaN,insuficiente,NORMAL
4,0011030010,CERTEGUI,CHOCO,CÉRTEGUI,ATRATO - DARIÉN,5.380,-76.610000,17.8,4.7,144,11030010,CRÍTICA,0.282051,0.421173,1.493251,estable,MODERADA


In [10]:
# ── Helpers robustos ante NaN ─────────────────────────────────────────────
def modo_seguro(serie, default='BAJA'):
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

def modo_sequia(serie):
    return modo_seguro(serie, default='NORMAL')

NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}

# Sjoin estaciones del día → municipios
gdf_dia = gpd.GeoDataFrame(
    df_dia,
    geometry=gpd.points_from_xy(df_dia.longitud, df_dia.latitud),
    crs="EPSG:4326",
).to_crs(epsg=3116)

df_muni = gpd.sjoin_nearest(gdf_mun, gdf_dia, how="left", distance_col="dist_m")

# Agrupación a nivel municipal
df_muni = (
    df_muni
    .groupby(["COD_MPIO", "NOM_MPIO", "NOM_DPTO", "LATITUD", "LONGITUD"], as_index=False)
    .agg(
        precip_acum_mm=("precip_acum_mm", "mean"),
        precip_max_10min=("precip_max_10min", "max"),
        n_estaciones=("codigoestacion", "count"),
        alerta_compuesta=("alerta_lluvias", modo_seguro),
        frecuencia_extremos=("frecuencia_extremos", "max"),
        frecuencia_reciente=("frecuencia_reciente", "max"),
        ratio_reciente=("ratio_reciente", "max"),
        tendencia=("tendencia", tendencia_muni),
        sequia_categoria=("sequia_categoria", modo_sequia),
    )
)

# alerta numérico derivado de alerta_compuesta (moda) → consistencia garantizada
df_muni["alerta"] = df_muni["alerta_compuesta"].map(NIVEL_NUMERICO).fillna(0).astype(int)
df_muni["fecha"]  = fecha_mapa

# ── Agrega flag de afectación histórica por inundaciones ──────────────────
flood = pd.read_excel(
    os.path.join(BASE_DIR, "Datos Procesados", "municipios_afectados_ola_invernal.xlsx"),
    usecols=['cod_divipola', 'municipio_afectado']
)
flood['cod_divipola'] = flood['cod_divipola'].astype(str)
df_muni['COD_MPIO'] = df_muni['COD_MPIO'].astype(str)
df_muni = df_muni.merge(flood, left_on='COD_MPIO', right_on='cod_divipola', how='left').drop(columns='cod_divipola')
df_muni['municipio_afectado'] = df_muni['municipio_afectado'].fillna(0).astype(int)

print(f"Municipios con datos   : {len(df_muni):,}")
print(f"\n=== Alerta compuesta por municipio (nivel dominante) ===")
print(df_muni['alerta_compuesta'].value_counts())
print(f"\nTotal con alerta > BAJA: {(df_muni['alerta'] > 0).sum():,}")
print(f"Municipios con antecedente inundación: {(df_muni['municipio_afectado'] == 1).sum():,}")

out_path = os.path.join(WEB_DATA_DIR, "datos_municipios.csv")
df_muni.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\nGuardado:", out_path)
df_muni.sort_values('alerta', ascending=False).head(5)

Municipios con datos   : 1,121

=== Alerta compuesta por municipio (nivel dominante) ===
alerta_compuesta
BAJA        786
MODERADA    284
ALTA         35
CRÍTICA      16
Name: count, dtype: int64

Total con alerta > BAJA: 335
Municipios con antecedente inundación: 753

Guardado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data\datos_municipios.csv


,COD_MPIO,NOM_MPIO,NOM_DPTO,LATITUD,LONGITUD,precip_acum_mm,precip_max_10min,n_estaciones,alerta_compuesta,frecuencia_extremos,frecuencia_reciente,ratio_reciente,tendencia,sequia_categoria,alerta,fecha,municipio_afectado
398,19807,TIMBÍO,CAUCA,2.369625,-76.710519,0.0,0.0,4,CRÍTICA,1.000000,1.000000,4.366152,decreciente,NORMAL,3,2026-04-05,1
378,19397,LA VEGA,CAUCA,2.053933,-76.758806,0.0,0.0,4,CRÍTICA,1.000000,1.000000,4.366152,decreciente,NORMAL,3,2026-04-05,1
391,19693,SAN SEBASTIÁN,CAUCA,1.856262,-76.716524,0.0,0.0,4,CRÍTICA,1.000000,1.000000,4.366152,decreciente,NORMAL,3,2026-04-05,1
377,19392,LA SIERRA,CAUCA,2.187474,-76.782280,0.0,0.0,4,CRÍTICA,1.000000,1.000000,4.366152,decreciente,NORMAL,3,2026-04-05,1
198,15051,ARCABUCO,BOYACÁ,5.735264,-73.427931,0.3,0.1,3,CRÍTICA,0.148571,0.166327,7.720330,decreciente,NORMAL,3,2026-04-05,0


## GeoJSON de municipios con polígonos para el dashboard

In [ ]:

# ── Genera GeoJSON de municipios con polígonos para el dashboard ─────────────
import requests, zipfile, io

# ── 1. Geometría (cache local; solo se descarga una vez) ─────────────────────
geo_cache = os.path.join(WEB_DATA_DIR, 'municipios_colombia_geo.gpkg')

if os.path.exists(geo_cache):
    print("Cargando geometría desde caché local...")
    gdf_geo = gpd.read_file(geo_cache)
else:
    print("Descargando polígonos GADM Colombia nivel 2 (primera vez)...")
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_COL_2.json.zip"
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        fname = next(f for f in z.namelist() if f.endswith('.json'))
        with z.open(fname) as f:
            gdf_geo = gpd.read_file(f)
    gdf_geo = gdf_geo.to_crs('EPSG:4326')
    gdf_geo['geometry'] = gdf_geo['geometry'].simplify(tolerance=0.005, preserve_topology=True)
    gdf_geo = gdf_geo[['NAME_1', 'NAME_2', 'geometry']].copy()
    gdf_geo.to_file(geo_cache, driver='GPKG')
    print(f"  → {len(gdf_geo):,} polígonos guardados en caché ({geo_cache})")

# ── Asignar COD_MPIO (código DANE) a polígonos GADM si aún no está ───────────
if 'COD_MPIO' not in gdf_geo.columns:
    print("Asignando códigos DANE a polígonos GADM via sjoin...")
    gdf_mun_4326 = gdf_mun.to_crs('EPSG:4326')[['COD_MPIO', 'geometry']].copy()

    # Cada punto DANE cae dentro de su polígono GADM
    joined = gpd.sjoin(gdf_mun_4326, gdf_geo[['geometry']], how='left', predicate='within')
    code_map = (
        joined.dropna(subset=['index_right'])
        .groupby('index_right')['COD_MPIO']
        .first()
    )
    gdf_geo['COD_MPIO'] = gdf_geo.index.map(code_map)

    # Polígonos sin coincidencia → asignar por el punto más cercano
    missing = gdf_geo['COD_MPIO'].isna()
    if missing.any():
        near = gpd.sjoin_nearest(
            gdf_geo[missing][['geometry']].reset_index(),
            gdf_mun_4326.reset_index(drop=True),
            how='left',
        ).drop_duplicates('index').set_index('index')['COD_MPIO']
        gdf_geo.loc[missing, 'COD_MPIO'] = gdf_geo[missing].index.map(near)

    gdf_geo['COD_MPIO'] = gdf_geo['COD_MPIO'].astype(str).str.strip()
    gdf_geo.to_file(geo_cache, driver='GPKG')
    print(f"  → COD_MPIO asignado ({gdf_geo['COD_MPIO'].notna().sum():,} polígonos) y caché actualizado")

print(f"Polígonos disponibles: {len(gdf_geo):,}")

# ── 2. Preparar datos de alerta (df_muni está en memoria del paso anterior) ──
df_alert = df_muni.copy()
df_alert['COD_MPIO'] = df_alert['COD_MPIO'].astype(str).str.strip()

cols_join = [
    'COD_MPIO', 'NOM_MPIO', 'NOM_DPTO',
    'alerta', 'alerta_compuesta', 'precip_acum_mm', 'precip_max_10min',
    'n_estaciones', 'sequia_categoria', 'fecha', 'municipio_afectado',
    'frecuencia_extremos', 'frecuencia_reciente', 'tendencia',
]
df_alert = df_alert[[c for c in cols_join if c in df_alert.columns]]

# ── 3. Join por código DANE ──────────────────────────────────────────────────
gdf_geo['COD_MPIO'] = gdf_geo['COD_MPIO'].astype(str).str.strip()
gdf_out = gdf_geo.merge(df_alert, on='COD_MPIO', how='left')
matched = gdf_out['alerta'].notna().sum()
print(f"Polígonos con datos de alerta: {matched:,} / {len(gdf_out):,}")

# ── 4. Rellenar faltantes ─────────────────────────────────────────────────────
for col in ['precip_acum_mm', 'precip_max_10min', 'frecuencia_extremos', 'frecuencia_reciente', 'n_estaciones']:
    gdf_out[col] = pd.to_numeric(gdf_out[col], errors='coerce').fillna(0)
gdf_out['alerta']             = pd.to_numeric(gdf_out['alerta'], errors='coerce').fillna(-1).astype(int)
gdf_out['alerta_compuesta']   = gdf_out['alerta_compuesta'].fillna('SIN_DATOS')
gdf_out['sequia_categoria']   = gdf_out['sequia_categoria'].fillna('NORMAL')
gdf_out['tendencia']          = gdf_out['tendencia'].fillna('sin_datos')
gdf_out['municipio_afectado'] = pd.to_numeric(gdf_out['municipio_afectado'], errors='coerce').fillna(0).astype(int)
gdf_out['NOM_MPIO']           = gdf_out['NOM_MPIO'].fillna(gdf_out['NAME_2'])
gdf_out['NOM_DPTO']           = gdf_out['NOM_DPTO'].fillna(gdf_out['NAME_1'])

# ── 5. Exportar GeoJSON ───────────────────────────────────────────────────────
keep = ['NAME_1', 'NAME_2', 'COD_MPIO', 'NOM_MPIO', 'NOM_DPTO',
        'alerta', 'alerta_compuesta', 'precip_acum_mm', 'precip_max_10min',
        'n_estaciones', 'sequia_categoria', 'fecha', 'municipio_afectado',
        'frecuencia_extremos', 'frecuencia_reciente', 'tendencia', 'geometry']
gdf_final = gdf_out[[c for c in keep if c in gdf_out.columns]]

out_geo = os.path.join(WEB_DATA_DIR, 'municipios_alertas.geojson')
gdf_final.to_file(out_geo, driver='GeoJSON')
size_kb = os.path.getsize(out_geo) / 1024

print(f"\nGeoJSON exportado: {out_geo}")
print(f"  Tamaño        : {size_kb:.0f} KB")
print(f"  ALTA o CRÍTICA: {(gdf_final['alerta'] >= 2).sum():,} municipios")
print(f"  Sequía severa : {(gdf_final['sequia_categoria'].isin(['SEVERA','EXTREMA'])).sum():,} municipios")
gdf_final.sort_values('alerta', ascending=False).head(5)[['NOM_MPIO','NOM_DPTO','alerta_compuesta','precip_acum_mm']]
